# Refresh `model_benchmarks` — Artificial Analysis (Premium)

**Daily** job. Pulls the full **`/language/models`** (Pro) catalog and **overwrites** `model_benchmarks`.
The premium endpoint gives cost-per-task, a real open-weights flag, a reasoning flag, MoE params, and
richer evals directly — so this replaces the old `/data/llms/models` + LLM-Stats merge.

The biweekly `model_benchmarks_history` table is **retired** (dropped below) — no longer needed.

**Prereq:** secret `AA_API_KEY` in scope `frontier_labs` must be the **Pro** key (`aa_…`).

In [ ]:
CATALOG, SCHEMA, SCOPE, AA_SECRET = "fso_market_intelligence", "frontier_labs", "frontier_labs", "AA_API_KEY"
BASE = "https://artificialanalysis.ai/api/v2/language/models"

import requests, time
AA_KEY = dbutils.secrets.get(SCOPE, AA_SECRET)

def fetch_all():
    out, page = [], 1
    while True:
        for a in range(4):
            r = requests.get(BASE, headers={"x-api-key": AA_KEY, "Accept": "application/json"},
                             params={"page": page, "page_size": 200}, timeout=60)
            if r.status_code == 429:
                time.sleep(15 * (a + 1)); continue
            r.raise_for_status(); break
        j = r.json(); out += j.get("data", [])
        pg = j.get("pagination", {})
        if not pg.get("has_more"):
            print(f"tier={j.get('tier')} intelligence_index_version={j.get('intelligence_index_version')}")
            break
        page += 1; time.sleep(0.5)
    return out

aa = fetch_all()
print("models:", len(aa))

In [ ]:
# ── flatten to the wide model_benchmarks schema ──
def num(x):
    try: return float(x) if x is not None else None
    except (TypeError, ValueError): return None

rows = []
for m in aa:
    cr = m.get("model_creator") or {}
    lic = m.get("licensing") or {}
    pa = m.get("parameters") or {}
    ev = m.get("evaluations") or {}
    pr = m.get("pricing") or {}
    perf = m.get("performance") or {}
    cost = m.get("artificial_analysis_intelligence_index_cost") or {}
    cpt = (cost.get("cost_per_task") or {})
    rows.append({
        "id": m.get("id"), "name": m.get("name"), "slug": m.get("slug"),
        "org": cr.get("name"), "country": cr.get("country"), "release_date": m.get("release_date"),
        "is_open_weights": lic.get("is_open_weights"), "reasoning_model": m.get("reasoning_model"),
        "params_total_b": num(pa.get("total")), "params_active_b": num(pa.get("active")),
        "context_window_tokens": int(m["context_window_tokens"]) if m.get("context_window_tokens") else None,
        # pricing ($/1M tokens)
        "price_input": num(pr.get("price_1m_input_tokens")), "price_output": num(pr.get("price_1m_output_tokens")),
        "price_blended_3to1": num(pr.get("price_1m_blended_3_to_1")),
        "price_blended_7to2to1": num(pr.get("price_1m_blended_7_to_2_to_1")),
        "price_cache_hit": num(pr.get("price_1m_cache_hit_tokens")),
        # cost per Intelligence-Index task ($)
        "cost_per_task_total": num(cpt.get("total_cost")),
        # evals
        "intelligence_index": num(ev.get("artificial_analysis_intelligence_index")),
        "coding_index": num(ev.get("artificial_analysis_coding_index")),
        "agentic_index": num(ev.get("artificial_analysis_agentic_index")),
        "multilingual_index": num(ev.get("artificial_analysis_multilingual_index")),
        "openness_index": num(ev.get("artificial_analysis_openness_index")),
        "gpqa_diamond": num(ev.get("gpqa_diamond")), "hle": num(ev.get("hle")),
        "terminalbench_hard": num(ev.get("terminalbench_hard")), "mmmu_pro": num(ev.get("mmmu_pro")),
        "scicode": num(ev.get("scicode")), "ifbench": num(ev.get("ifbench")),
        "aa_lcr": num(ev.get("aa_lcr")),
        # speed
        "tokens_per_sec": num(perf.get("median_output_tokens_per_second")),
        "ttft": num(perf.get("median_time_to_first_token_seconds")),
    })
print("rows:", len(rows), "| with cost_per_task:", sum(1 for r in rows if r['cost_per_task_total'] is not None))

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, DoubleType, LongType

S = lambda n, t: StructField(n, t())
schema = StructType([
    S("id", StringType), S("name", StringType), S("slug", StringType), S("org", StringType),
    S("country", StringType), S("release_date", StringType), S("is_open_weights", BooleanType),
    S("reasoning_model", BooleanType), S("params_total_b", DoubleType), S("params_active_b", DoubleType),
    S("context_window_tokens", LongType), S("price_input", DoubleType), S("price_output", DoubleType),
    S("price_blended_3to1", DoubleType), S("price_blended_7to2to1", DoubleType), S("price_cache_hit", DoubleType),
    S("cost_per_task_total", DoubleType), S("intelligence_index", DoubleType), S("coding_index", DoubleType),
    S("agentic_index", DoubleType), S("multilingual_index", DoubleType), S("openness_index", DoubleType),
    S("gpqa_diamond", DoubleType), S("hle", DoubleType), S("terminalbench_hard", DoubleType),
    S("mmmu_pro", DoubleType), S("scicode", DoubleType), S("ifbench", DoubleType), S("aa_lcr", DoubleType),
    S("tokens_per_sec", DoubleType), S("ttft", DoubleType),
])

df = (spark.createDataFrame(rows, schema=schema)
      .withColumn("release_date", F.to_date("release_date"))
      .withColumn("captured_at", F.current_date()))
(df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.model_benchmarks"))
print("model_benchmarks rows:", spark.table(f"{CATALOG}.{SCHEMA}.model_benchmarks").count())

# retire the old biweekly history table (no longer maintained)
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.model_benchmarks_history")
print("dropped model_benchmarks_history")

display(spark.table(f"{CATALOG}.{SCHEMA}.model_benchmarks").orderBy(F.col("intelligence_index").desc_nulls_last()).limit(10))

## Scheduling
One **daily** Job (Workflows → this notebook, **Source = Git provider / branch `main`**, serverless). The
run-as identity needs **READ** on the `frontier_labs` secret scope. Overwrites `model_benchmarks` each run.